# Bose2 — Free no-GPU cinematic crochet montage

Create a narrated 1080p crochet video from clips you own or are licensed to use. This CPU/FFmpeg workflow adds Egyptian Arabic narration with Edge TTS and burns Arabic captions. No GPU, paid API, billing, face/avatar, or publishing is involved.

**Important:** Edge TTS requires internet access and may be temporarily unavailable. Do not upload copyrighted clips unless you have permission. Review the exported video before sharing.

In [ ]:
!git clone --depth 1 --branch feature/real-hands-footage-mode https://github.com/bosesamer-code/Bose2.git /content/Bose2
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core >/dev/null
!python -m pip -q install edge-tts
import sys, subprocess
from pathlib import Path
sys.path.insert(0, '/content/Bose2')
from src.media_providers import MediaRequest
from src.hybrid_cinematic_providers import HybridNarrationProvider
from src.real_footage_providers import RealFootageVideoProvider
print('Setup complete. This workflow does not require a GPU or paid API.')

## 1. Upload crochet footage

Upload one or more MP4, MOV, MKV, or WEBM clips. Use your own footage or media you have permission to use.

In [ ]:
from google.colab import files
uploaded = files.upload()
input_dir = Path('/content/bose2_clips')
input_dir.mkdir(parents=True, exist_ok=True)
clips = []
for name, data in uploaded.items():
    path = input_dir / Path(name).name
    path.write_bytes(data)
    if path.suffix.lower() in {'.mp4', '.mov', '.mkv', '.webm'} and path.stat().st_size:
        clips.append(path)
if not clips:
    raise ValueError('Upload at least one valid video clip (MP4/MOV/MKV/WEBM).')
print(f'Loaded {len(clips)} clip(s).')

## 2. Write the Egyptian Arabic narration and captions

Edit `script` and the caption text. Captions are scene-based, so keep each scene caption short and clear. The voice is `ar-EG-SalmaNeural`.

In [ ]:
script = '''أهلًا بيكم! النهارده هنتعلم خطوة بسيطة في الكروشيه. جهزوا الخيط والسنارة، وخلينا نبدأ واحدة واحدة. ركزوا في حركة الإيد، وكرروا الخطوة بهدوء لحد ما تتقنوها.'''
scene_plan = [
    {'scene': 1, 'purpose': 'hook', 'duration_seconds': 8, 'text': 'يلا نتعلم خطوة جديدة في الكروشيه'},
    {'scene': 2, 'purpose': 'lesson', 'duration_seconds': 12, 'text': 'جهزي الخيط والسنارة وابدئي بهدوء'},
    {'scene': 3, 'purpose': 'application', 'duration_seconds': 12, 'text': 'كرري الحركة لحد ما تتقنيها'},
]
if not script.strip():
    raise ValueError('Write the narration script before rendering.')
print('Narration and captions ready. Check the Arabic wording before the next cell.')

## 3. Render the video

The free TTS service needs internet. Rendering is CPU-based and can take a while. If it fails, the notebook stops instead of creating a silent video.

In [ ]:
job_dir = Path('/content/bose2_montage_job')
job_dir.mkdir(parents=True, exist_ok=True)
audio_result = HybridNarrationProvider().generate(MediaRequest('crochet-montage', job_dir / 'audio', {'script': script, 'voice': 'ar-EG-SalmaNeural'}))
if audio_result.status != 'completed':
    raise RuntimeError(audio_result.error or 'Egyptian Arabic narration failed.')
video_result = RealFootageVideoProvider().generate(MediaRequest('crochet-montage', job_dir / 'video', {'audio_path': audio_result.artifacts[0], 'real_footage_paths': [str(p) for p in clips], 'scene_plan': scene_plan}))
if video_result.status != 'completed':
    raise RuntimeError(video_result.error or 'Video render failed.')
video_path = Path(video_result.artifacts[0])
print('Video ready:', video_path, 'size:', video_path.stat().st_size, 'bytes')

In [ ]:
from google.colab import files
files.download(str(video_path))
print('Watch the entire MP4 and check narration, captions, crop, and media rights before sharing. Nothing was uploaded or published.')